# BÁO CÁO KIỂM THỬ TỰ ĐỘNG - ĐỘI BUILD MODEL

**Mục tiêu:** Script này của Đội Testing sẽ tự động đọc file `build_model.ipynb` và kiểm tra 3 lỗi logic quan trọng.

**Quy trình:**
1.  Đọc file `.ipynb` của đội Model.
2.  Test 1: Kiểm tra Nguồn dữ liệu (phải là file đã duyệt).
3.  Test 2: Kiểm tra Rò rỉ dữ liệu (không được `fit` trên tập Test).
4.  Test 3: Kiểm tra Chỉ số Đánh giá (không được dùng 'Accuracy' làm chính).

## 1. Khai báo thư viện và Tên file

Khai báo các thư viện cần thiết và tên các file "đúng" và file "sai".

In [3]:
import json # Để đọc file .ipynb
import sys  # Dùng cho các hàm hệ thống

# --- TÊN FILE CẦN KIỂM TRA ---
FILE_NOTEBOOK_MODEL = 'build_model.ipynb'

# Các file đúng (phải xuất hiện)
FILE_TRAIN_CSV = 'Dữ liệu sau tiền xử lí train.csv'
FILE_TEST_CSV = 'Dữ liệu sau tiền xử lí test.csv'

# File sai (không được xuất hiện)
FILE_RAW_CSV = 'File dữ liệu thô.csv'

print(">>> Các biến đã được khai báo.")

>>> Các biến đã được khai báo.


## 2. Tải và Đọc file .ipynb

Đọc file .ipynb như một file JSON và trích xuất toàn bộ code.

In [4]:
try:
    with open(FILE_NOTEBOOK_MODEL, 'r', encoding='utf-8') as f:
        notebook_data = json.load(f)
except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file '{FILE_NOTEBOOK_MODEL}'.")
    # sys.exit() # (bỏ sys.exit để notebook không bị crash)

# Nối tất cả code trong các ô "code" lại thành 1 chuỗi lớn
full_code_string = ""
for cell in notebook_data['cells']:
    if cell['cell_type'] == 'code':
        # Thêm 1 dòng mới để các lệnh không dính vào nhau
        full_code_string += "\n".join(cell['source']) + "\n"

print(">>> Đã đọc xong file code của đội Model. Bắt đầu kiểm tra...")
# print(full_code_string) # Bỏ comment này để xem toàn bộ code

>>> Đã đọc xong file code của đội Model. Bắt đầu kiểm tra...


## 3. Bài Test 1: Kiểm tra Nguồn Dữ liệu

Kiểm tra xem đội Model có dùng đúng 2 file `...train.csv` và `...test.csv` mà chúng ta đã duyệt không.

In [5]:
print("--- [Test 1]: Kiểm tra Nguồn Dữ liệu ---")
test1_passed = True

if FILE_TRAIN_CSV not in full_code_string or FILE_TEST_CSV not in full_code_string:
    print(f"  ❌ FAILED: Không tìm thấy code đọc file '{FILE_TRAIN_CSV}' hoặc '{FILE_TEST_CSV}'.")
    test1_passed = False

if FILE_RAW_CSV in full_code_string:
    print(f"  ❌ FAILED: Phát hiện code đọc file thô '{FILE_RAW_CSV}'. Lỗi logic.")
    test1_passed = False

if test1_passed:
    print("  ✅ PASSED: Đã sử dụng đúng 2 file train/test đã được duyệt.")

--- [Test 1]: Kiểm tra Nguồn Dữ liệu ---
  ✅ PASSED: Đã sử dụng đúng 2 file train/test đã được duyệt.


## 4. Bài Test 2: Kiểm tra Rò rỉ dữ liệu khi Huấn luyện

Kiểm tra xem mô hình có bị `fit` (huấn luyện) nhầm trên `X_test` hay `X` (toàn bộ) không.

In [6]:
print("\n--- [Test 2]: Kiểm tra Quy trình Huấn luyện (Rò rỉ) ---")
test2_passed = True
# Loại bỏ khoảng trắng để dễ tìm
code_no_spaces = full_code_string.replace(" ", "")

if ".fit(X_test" in code_no_spaces:
    print("  ❌ FAILED: Phát hiện model.fit() trên 'X_test'. Lỗi rò rỉ nghiêm trọng.")
    test2_passed = False
elif ".fit(X," in code_no_spaces and ".fit(X_train" not in code_no_spaces:
     print("  ❌ FAILED: Phát hiện model.fit() trên 'X' (toàn bộ data). Lỗi rò rỉ.")
     test2_passed = False
elif ".fit(X_train,y_train)" not in code_no_spaces:
    print("  ❌ FAILED: Không tìm thấy lệnh .fit(X_train, y_train) chuẩn.")
    test2_passed = False

if test2_passed:
    print("  ✅ PASSED: Quy trình huấn luyện (fit) đúng trên tập Train.")


--- [Test 2]: Kiểm tra Quy trình Huấn luyện (Rò rỉ) ---
  ✅ PASSED: Quy trình huấn luyện (fit) đúng trên tập Train.


## 5. Bài Test 3: Kiểm tra Chỉ số Đánh giá (Metrics)

Kiểm tra xem đội Model có rơi vào "bẫy Accuracy" không. Yêu cầu bắt buộc là phải báo cáo `confusion_matrix` hoặc `classification_report`.

In [7]:
print("\n--- [Test 3]: Kiểm tra Chỉ số Đánh giá (Bẫy Accuracy) ---")
test3_passed = True

# Tìm kiếm các từ khóa
has_matrix = "confusion_matrix" in full_code_string
has_report = "ClassificationReport" in full_code_string or "classification_report" in full_code_string
has_accuracy = "accuracy_score" in full_code_string

if not has_matrix and not has_report:
    print("  ❌ FAILED: Không báo cáo 'confusion_matrix' hoặc 'classification_report'.")
    print("            Đây là yêu cầu bắt buộc vì dữ liệu test bị mất cân bằng.")
    test3_passed = False
elif has_accuracy and not has_matrix and not has_report:
     print("  ❌ FAILED: Chỉ báo cáo 'accuracy_score' là không đủ và gây hiểu nhầm.")
     test3_passed = False
     
if test3_passed:
    print("  ✅ PASSED: Đã sử dụng các chỉ số (Matrix, Report) phù hợp.")

print("\n--- KẾT THÚC KIỂM THỬ ---")


--- [Test 3]: Kiểm tra Chỉ số Đánh giá (Bẫy Accuracy) ---
  ✅ PASSED: Đã sử dụng các chỉ số (Matrix, Report) phù hợp.

--- KẾT THÚC KIỂM THỬ ---
